In [ ]:
%pip install nba_api
%pip install requests
%pip install requests-cache
%pip install xgboost
%pip install shap cupy-cuda12x
%pip install beautifulsoup4
%pip install optuna
%pip install pytz

In [ ]:
import pandas as pd
from tqdm import tqdm
from requests import Request
from nba_api.stats.endpoints import boxscoreadvancedv3
from nba_api.stats.library.http import NBAStatsHTTP
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.live.nba.endpoints import scoreboard
from sklearn.preprocessing import MinMaxScaler
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupKFold
from sklearn import set_config
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
import shap
import xgboost as xgb
import random
import time
import math
import json
import requests
import requests_cache
import numpy as np
import unicodedata
import pytz
from IPython.display import display

In [ ]:
def removeSuffix(s):
  split = s.strip().split(" ")
  out = split[0] + " " + split[1]
  suffixes = ['Jr.', 'Sr.', 'II', 'III', 'IV']
  for i in range(2, len(split)):
    if not split[i] in suffixes:
      out = out + " " + split[i]
  return out

def strip(s):
  normalized = unicodedata.normalize('NFD', s)
  stripped = ''.join(c for c in normalized if unicodedata.category(c) != 'Mn')

  return removeSuffix(stripped)

last_call = 0
MIN_INTERVAL = 1.2
def rate_limited_call(game_id):
  global last_call
  url = f"https://stats.nba.com/stats/boxscoreadvancedv3?EndPeriod=0&EndRange=0&GameID={game_id}&RangeType=0&StartPeriod=0&StartRange=0"
  req = Request('GET', url).prepare()
  key = NBAStatsHTTP._session.cache.create_key(req)

  cached_response = NBAStatsHTTP._session.cache.get_response(key)
  if cached_response is not None:
        return cached_response.json()

  elapsed = time.time() - last_call
  if elapsed < MIN_INTERVAL:
      time.sleep(MIN_INTERVAL - elapsed)

  result = boxscoreadvancedv3.BoxScoreAdvancedV3(game_id=game_id).get_dict()

  last_call = time.time()
  return result

def WNI(pie, mins, usg, comment): # calculates a players impact in a given game
  if mins:
    mins = float(mins.split(':')[0]) + float(mins.split(':')[1])/60
  else:
    if comment[:3]=="DNP": # handles DNP/DND context for player impact
      mins = 0
    else:
      return np.nan
  return mins * usg * pie

def add_rolling(df, group_cols, value_col, windows, prefix):
    for w in windows:
        df[f"{prefix}{w}_rolling_{value_col}"] = (
            df.groupby(group_cols)[value_col]
              .rolling(window=w, min_periods=1)
              .mean()
              .reset_index(level=list(range(len(group_cols))), drop=True)
        )
    return df

def find_weighted_team_averages(team, span, context, cols):
  team = team.copy()
  if context==1:
    homeMasked = team[cols].where(team['home'] == 1)
    awayMasked = team[cols].where(team['home'] == 0)

    ewmaHome = homeMasked.ewm(span=span, adjust=False).mean()
    ewmaAway = awayMasked.ewm(span=span, adjust=False).mean()

    out = ewmaHome.where(team['next_home'] == 1, ewmaAway)
    out = out.mask(team['next_home'].isna())
  else:
    out = team[cols].ewm(span=span, adjust=False).mean()
  return out


def computeStreak(group):
    streak = 0
    streak_list = []
    for result in group['WL']:
        if result == 1:
            streak = streak + 1 if streak >= 0 else 1
        else:
            streak = streak - 1 if streak <= 0 else -1
        streak_list.append(streak)
    # Return a Series with the same index
    return pd.Series(streak_list, index=group.index)

def computeRecord(group):
    wins = 0
    losts = 0
    record_list = []
    for result in group['WL']:
      if result == 1:
        wins+=1
      else:
        losts+=1
      record_list.append(wins/(wins+losts))
    return pd.Series(record_list, index = group.index)

def get_rotowire_lineups():
    headers = {"User-Agent": "Mozilla/5.0"}
    cache_buster = int(time.time())
    url = f"https://www.rotowire.com/basketball/nba-lineups.php?t={cache_buster}"
    headers = {"User-Agent": "Mozilla/5.0"}

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")
    date = soup.find("div", class_="page-title__secondary")
    game_blocks = soup.find_all("div", class_="lineup__box")
    date_text = soup.find("div", class_="page-title__secondary").get_text(strip=True)
    prefix = len("Starting lineups for ")
    date = date_text[prefix:]
    games = []
    for game in game_blocks:
        try:
            # --- Teams ---

            teams = game.find_all("a", class_=("lineup__team"))

            away = teams[0].find("div", class_="lineup__abbr").get_text(strip=True)
            home = teams[1].find("div", class_="lineup__abbr").get_text(strip=True)

            lineup = game.find("div", class_="lineup__main")
            awayLineup = lineup.select_one("ul.lineup__list.is-visit")
            awayStatus = awayLineup.find("li", class_="lineup__status").get_text(strip=True)
            awayPlayers = awayLineup.find_all("li", class_="lineup__player")
            awayStartingFive = []
            homeLineup = lineup.select_one("ul.lineup__list.is-home")
            homeStatus = homeLineup.find("li", class_="lineup__status").get_text(strip=True)
            homePlayers = homeLineup.find_all("li", class_="lineup__player")
            homeStartingFive = []

            awayLineupStrength = 0
            homeLineupStrength = 0
            for player in awayPlayers[:5]:
              name = player.find("a").get("title")
              awayStartingFive.append(name)
            for player in homePlayers[:5]:
              name = player.find("a").get("title")
              homeStartingFive.append(name)
            games.append(
                {
                    "matchup": f"{away} @ {home}",
                    "away": away,
                    "home": home,
                    "awayStatus": awayStatus,
                    "homeStatus": homeStatus,
                    "awayLineup": awayStartingFive,
                    "homeLineup": homeStartingFive,
                    "date": date
                }
            )
        except Exception as e:
            print("Error parsing a game:", e)
    return pd.DataFrame(games)

In [ ]:
teams = ['ATL', 'BOS', 'BKN', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU',
         'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 'OKC', 'ORL',
         'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS']

# seasons = ["2016-17", "2017-18", "2018-19", "2019-20", "2020-21", "2021-22", "2022-23", "2023-24", "2024-25", "2025-26"]
seasons = ['2025-26']
drop_cols = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID']
frames = []

# caching/rate limiting
requests_cache.install_cache('nba_api_cache', expire_after=7*24*3600)
NBAStatsHTTP._session = requests_cache.CachedSession('nba_api_cache', backend='sqlite')
#get boxscores across every season in seasons
for season in seasons:
    url = f"https://stats.nba.com/stats/leaguegamefinder?Conference=&DateFrom=&DateTo=&Division=&DraftNumber=&DraftRound=&DraftTeamID=&DraftYear=&EqAST=&EqBLK=&EqDD=&EqDREB=&EqFG3A=&EqFG3M=&EqFG3_PCT=&EqFGA=&EqFGM=&EqFG_PCT=&EqFTA=&EqFTM=&EqFT_PCT=&EqMINUTES=&EqOREB=&EqPF=&EqPTS=&EqREB=&EqSTL=&EqTD=&EqTOV=&GameID=&GtAST=&GtBLK=&GtDD=&GtDREB=&GtFG3A=&GtFG3M=&GtFG3_PCT=&GtFGA=&GtFGM=&GtFG_PCT=&GtFTA=&GtFTM=&GtFT_PCT=&GtMINUTES=&GtOREB=&GtPF=&GtPTS=&GtREB=&GtSTL=&GtTD=&GtTOV=&LeagueID=00&Location=&LtAST=&LtBLK=&LtDD=&LtDREB=&LtFG3A=&LtFG3M=&LtFG3_PCT=&LtFGA=&LtFGM=&LtFG_PCT=&LtFTA=&LtFTM=&LtFT_PCT=&LtMINUTES=&LtOREB=&LtPF=&LtPTS=&LtREB=&LtSTL=&LtTD=&LtTOV=&Outcome=&PORound=&PlayerID=&PlayerOrTeam=T&RookieYear=&Season={season}&SeasonSegment=&SeasonType=Regular+Season&StarterBench=&TeamID=&VsConference=&VsDivision=&VsTeamID=&YearsExperience="
    req = Request('GET', url).prepare()
    key = NBAStatsHTTP._session.cache.create_key(req)
    cached_response = NBAStatsHTTP._session.cache.get_response(key)

    if cached_response is not None:
      response = cached_response.json()
      df = pd.DataFrame(response['resultSets'][0]['rowSet'], columns = response['resultSets'][0]['headers'])
    else:
      gamefinder = leaguegamefinder.LeagueGameFinder(
          league_id_nullable='00',
          season_nullable=season,
          season_type_nullable='Regular Season',
          date_to_nullable= (datetime.now(pytz.timezone('US/Pacific')) - timedelta(days=1)).strftime('%Y-%m-%d')
      )
      df = gamefinder.get_data_frames()[0]

    df = df.drop(columns=drop_cols)



    df.insert(3, "season", season)
    df.insert(4, "home", df["MATCHUP"].str.contains("vs").astype(int))
    df.insert(7, "target", None)
    df['WL'] = (df['WL'] == 'W').astype(int)

    frames.append(df)

box_df = pd.concat(frames, ignore_index=True)
box_df.reset_index(drop=True, inplace=True)
box_df.sort_values(by=['TEAM_ABBREVIATION', 'season', 'GAME_DATE'], ascending=[True, True, True], inplace=True)


all_data_list = []
all_player_list = []

all_team_rows = []
all_player_rows = []

for gameid, teamcode, date, season in tqdm(
    zip(box_df.GAME_ID, box_df.TEAM_ABBREVIATION, box_df.GAME_DATE, box_df.season)
):
    try:
        advanced_boxscore = rate_limited_call(gameid)

        if advanced_boxscore['boxScoreAdvanced']['awayTeam']['teamTricode'] == teamcode:
            team = advanced_boxscore['boxScoreAdvanced']['awayTeam']
            home = 0
        else:
            team = advanced_boxscore['boxScoreAdvanced']['homeTeam']
            home = 1
        team_stats = team['statistics']

        all_team_rows.append({
            **team_stats,
            "GAME_ID": gameid,
            "TEAM_ABBREVIATION": teamcode,
            "starters": [f"{p['firstName']} {p['familyName']}"
                                for p in team['players'][:5]],
        })
        for p in team['players']:
            stats = p["statistics"]
            player_name = f"{p['firstName']} {p['familyName']}"


            all_player_rows.append({
                "PLAYER_NAME": player_name,
                "GAME_ID": gameid,
                "GAME_DATE": date,
                "TEAM_ABBREVIATION": teamcode,
                "HOME": home,
                "WNI": WNI(stats['PIE'], stats['minutes'], stats['usagePercentage'], p['comment']),
            })

    except Exception as e:
        print(f"Error processing {gameid}: {e}")

# Build final DataFrames
advanced_df = pd.DataFrame(all_team_rows)
player_df   = pd.DataFrame(all_player_rows)

player_df = player_df.sort_values(by=['GAME_DATE'])
scraped_df = get_rotowire_lineups()

In [ ]:
player_df.to_json()

In [ ]:
df = box_df.merge(advanced_df, on=['GAME_ID', 'TEAM_ABBREVIATION'], how='left')
df = df.drop(columns=['minutes', 'estimatedOffensiveRating',
                      'estimatedDefensiveRating', 'estimatedNetRating',
                      'estimatedTeamTurnoverPercentage', 'usagePercentage',
                      'estimatedUsagePercentage', 'estimatedPace', 'REB', 'assistPercentage',
                     ])
df['idx'] = df['GAME_DATE'].astype(str) + '_' + df['TEAM_ABBREVIATION'].astype(str)
df.set_index('idx', inplace=True)

In [ ]:
for _, row in scraped_df.iterrows():

    GAME_DATE = datetime.strptime(row['date'], "%B %d, %Y").strftime("%Y-%m-%d")
    home = row['home']
    away = row['away']
    # HOME

    idx = f"{GAME_DATE}_{home}"
    df.at[idx, 'season'] = '2025-26'
    df.at[idx, 'MATCHUP'] = f"{home} vs. {away}"
    df.at[idx, 'home'] = 1
    df.at[idx, 'GAME_DATE'] = GAME_DATE
    df.at[idx, 'TEAM_ABBREVIATION'] = home
    df.at[idx, 'starters'] = row['homeLineup']


    # AWAY
    idx = f"{GAME_DATE}_{away}"

    df.at[idx, 'season'] = '2025-26'
    df.at[idx, 'MATCHUP'] = f"{away} @ {home}"
    df.at[idx, 'home'] = 0
    df.at[idx, 'GAME_DATE'] = GAME_DATE
    df.at[idx, 'TEAM_ABBREVIATION'] = away
    df.at[idx, 'starters'] = row['awayLineup']

In [ ]:
df['starters'] = df.groupby(['TEAM_ABBREVIATION', 'season'])['starters'].shift(-1)

In [ ]:
player_df = player_df.sort_values(["PLAYER_NAME", "HOME", "GAME_DATE"])

player_df = add_rolling(
    df=player_df,
    group_cols=["PLAYER_NAME", "HOME"],
    value_col="WNI",
    windows=[5, 10, 25],
    prefix="context_"
)
player_df = player_df.sort_values(["PLAYER_NAME", "GAME_DATE"])
player_df = add_rolling(
    df=player_df,
    group_cols=["PLAYER_NAME"],
    value_col="WNI",
    windows=[5, 10, 25],
    prefix=""
)

In [ ]:
df.reset_index(inplace=True, drop=True)
df = df.sort_values("GAME_DATE")
player_df = player_df.sort_values("GAME_DATE")

rolling_cols = [
    "context_5_rolling_WNI",
    "context_10_rolling_WNI",
    "context_25_rolling_WNI",
    "5_rolling_WNI",
    "10_rolling_WNI",
    "25_rolling_WNI"
]

games_exploded = df.explode("starters").rename(columns={"starters": "PLAYER_NAME"})

merged = games_exploded.merge(
    player_df[["PLAYER_NAME", "GAME_DATE"] + rolling_cols],
    on=["PLAYER_NAME", "GAME_DATE"],
    how="left"
)


starter_rolling_sum = merged.groupby(['GAME_ID', 'TEAM_ABBREVIATION'])[rolling_cols].sum()

df = df.merge(
    starter_rolling_sum.rename(columns=lambda x: f"lineup_{x}"),
    on=['GAME_ID', 'TEAM_ABBREVIATION'],
    how="left"
)

# Optional: make sure rows with missing starters are NaN
df.loc[df["starters"].isna(), [f"lineup_{col}" for col in rolling_cols]] = np.nan

In [ ]:
df['target'] = df.groupby(['TEAM_ABBREVIATION', 'season'])['WL'].shift(-1).astype('Int64')
df['next_home'] = df.groupby(['TEAM_ABBREVIATION', 'season'])['home'].shift(-1).astype('Int64')

df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'], format='%Y-%m-%d')
df['next_GAME_DATE'] = df.groupby(['TEAM_ABBREVIATION', 'season'])['GAME_DATE'].shift(-1)

df['rest_days'] = (df['next_GAME_DATE'] - df['GAME_DATE']).dt.days-1
df['recent_intensity'] = (
    df.groupby(['TEAM_ABBREVIATION', 'season'])['GAME_DATE']
      .transform(lambda x: 4 / ((x - x.shift(3)).dt.days + 1))
)
df['GAME_DATE'] = df['GAME_DATE'].dt.strftime('%Y-%m-%d')
df = df.drop(columns=['next_GAME_DATE'])

df['streak'] = 0
df['streak'] = df.groupby(['TEAM_ABBREVIATION', 'season'], group_keys=False).apply(computeStreak)
df['record'] = 0
df['record'] = df.groupby(['TEAM_ABBREVIATION', 'season'], group_keys=False).apply(computeRecord)
df.sort_values(by=['TEAM_ABBREVIATION', 'season', 'GAME_DATE'], ascending=[True, True, True], inplace=True)
df.reset_index(drop=True, inplace=True)


GameTotals = ['PTS', 'STL', 'BLK', 'TOV', 'AST', 'PF', 'FGM', 'FGA', 'FG3M', 'FG3A', 'FTA', 'FTM', 'DREB', 'OREB']
ppColumns = [f"pp_{col}" for col in GameTotals]
df[ppColumns] = df[GameTotals].div(df['possessions'], axis=0)
df.drop(columns=GameTotals, inplace=True)

In [ ]:
# Columns to exclude when computing weighted averages
removed_columns = ['GAME_DATE', 'MATCHUP', 'target', 'streak', 'next_home', 'WL',
                   'MIN', 'TEAM_ABBREVIATION', 'season', 'home', "context_5_rolling_PIE",
                  "lineup_context_5_rolling_PIE",
                  "lineup_context_10_rolling_PIE",
                  "lineup_context_25_rolling_PIE",
                  "lineup_5_rolling_PIE",
                  "lineup_10_rolling_PIE",
                  "lineup_25_rolling_PIE", 'record', 'starters']

selected_columns = df.columns[~df.columns.isin(removed_columns)]


# EWM configurations
configs = [
    (5, 1, "ewm5_context_", selected_columns),
    (10, 1, "ewm10_context_", selected_columns),
    (25, 1, "ewm25_context_", selected_columns),
    (5, 0, "ewm5_", selected_columns),
    (10, 0, "ewm10_", selected_columns),
    (25, 0, "ewm25_", selected_columns),
]

# Compute EWM features
copy = df.copy()
ewm_features = []

for span, context, prefix, cols in configs:
    temp = (
        copy
        .groupby(["TEAM_ABBREVIATION", "season"], group_keys=False)
        .apply(find_weighted_team_averages, span=span, context=context, cols=cols, include_groups=False)
        .add_prefix(prefix)
    )
    ewm_features.append(temp)

# Concatenate EWM features back to main dataframe
df = pd.concat([df] + ewm_features, axis=1)


In [ ]:
selected_columns = [column for column in df.columns if ('ewm' in column or 'lineup' in column)] + ['rest_days', 'recent_intensity', 'streak']
game_features = df[['TEAM_ABBREVIATION', 'GAME_DATE', 'season', 'GAME_ID']+selected_columns].copy()

game_features[selected_columns] = game_features.groupby(['TEAM_ABBREVIATION', 'season'])[selected_columns].shift(1)
game_features = game_features.rename(columns={c: f"opp_{c}" for c in selected_columns})

df['next_game_date'] =  df.groupby(['TEAM_ABBREVIATION', 'season'])['GAME_DATE'].shift(-1)
df['next_opp'] = df.groupby(['TEAM_ABBREVIATION', 'season'])['MATCHUP'].shift(-1).str[-3:]


result = df.merge(
    game_features,
    left_on=['next_game_date', 'next_opp'],
    right_on=['GAME_DATE', 'TEAM_ABBREVIATION'],
    how='left',
    suffixes=('', '_drop')
)


result = result.drop(columns=
 [
  'TEAM_ABBREVIATION_drop',
  'GAME_DATE_drop',
  'season_drop',
  'GAME_ID_drop',
  'next_opp',
  'next_game_date',
 ])
df = result.copy()


In [ ]:
df["25_context_net_rating_difference"] = df["ewm25_context_netRating"] - df["opp_ewm25_context_netRating"]
df["10_overall_net_rating_difference"] = df["ewm10_netRating"] - df["opp_ewm10_netRating"]
df["5_context_net_rating_difference"] = df["ewm5_context_netRating"] - df["opp_ewm5_context_netRating"]
df["25_overall_net_rating_difference"] = df["ewm25_netRating"] - df["opp_ewm25_netRating"]
df["10_context_net_rating_difference"] = df["ewm10_context_netRating"] - df["opp_ewm10_context_netRating"]
df["5_overall_net_rating_difference"] = df["ewm5_netRating"] - df["opp_ewm5_netRating"]
df["context_season_lineup_difference"] = df['lineup_context_25_rolling_WNI'] - df['opp_lineup_context_25_rolling_WNI']
df["context_month_lineup_difference"] = df['lineup_context_10_rolling_WNI'] - df['opp_lineup_context_10_rolling_WNI']
df["context_recent_lineup_difference"] = df["lineup_context_5_rolling_WNI"] - df['opp_lineup_context_5_rolling_WNI']
df["season_lineup_difference"] = df['lineup_25_rolling_WNI'] - df['opp_lineup_25_rolling_WNI']
df["month_lineup_difference"] = df['lineup_10_rolling_WNI'] - df['opp_lineup_10_rolling_WNI']
df["recent_lineup_difference"] = df["lineup_5_rolling_WNI"] - df['opp_lineup_5_rolling_WNI']
df['rest_difference'] = df['rest_days'] - df['opp_rest_days']

In [ ]:
removed_columns = ['TEAM_ABBREVIATION', 'GAME_ID', 'GAME_DATE', 'season', 'home', 'MATCHUP', 'WL', 'target', 'MIN','starters']
selected_columns = df.columns[~df.columns.isin(removed_columns)]

In [ ]:
df[selected_columns] = df[selected_columns].astype(float)
df = df.dropna(subset=selected_columns)
df.reset_index(drop=True, inplace=True)

In [ ]:
split = int(0.8*len(df))
df_sorted = df.sort_values(by=['GAME_DATE']).reset_index(drop=True).copy()

future_X = df[df['target'].isna()]

# X = df_sorted.iloc[:split][selected_columns]
# y = df_sorted.iloc[:split]['target']

# X_val = df_sorted.iloc[split:][selected_columns]
# X_val = cudf.from_pandas(X_val)
# y_val = df_sorted.iloc[split:]['target']
df_copy = df.dropna().copy()
df_copy.reset_index(drop=True, inplace=True)
X = df_copy[df_copy['season']!='2025-26'][selected_columns]
y = df_copy[df_copy['season']!='2025-26']['target']

X_val = df[df['season']=='2025-26'][selected_columns]
y_val = df[df['season']=='2025-26']['target']

model = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.5,
    eval_metric="auc",
    objective="binary:logistic",
    device="cuda",
    random_state = 42,
    n_jobs = -1
)

model.fit(X, y)

In [ ]:
background = X.sample(n=500, random_state=42)
sample = X.sample(n=8000, random_state=1337)
explainer = shap.TreeExplainer(
    model,
    data=background,
    feature_perturbation="auto"
)
shap_values = explainer.shap_values(sample)

In [ ]:
shap_importance = np.abs(shap_values).mean(axis=0)
indices = np.argsort(shap_importance)[::-1]
predictors = selected_columns[indices][:143]
# model.fit(X[predictors], y)
# y_pred_proba = model.predict_proba(X_val[predictors])[:, 1]
# y_pred = model.predict(X_val[predictors])
# roc_auc = roc_auc_score(y_val, y_pred_proba)
# accuracy = accuracy_score(y_val, y_pred)
# print(f"Accuracy: {accuracy}")
# print(f"ROC AUC: {roc_auc}")
predictors

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm.notebook import tqdm
maxacc = 0
maxroc = 0
accbest = 0
rocbest = 0
for i in tqdm(range(1, len(selected_columns)+1)):

  predictors = selected_columns[indices][:i]
  model.fit(X[predictors], y)
  y_pred_proba = model.predict_proba(X_val[predictors])[:, 1]
  y_pred = model.predict(X_val[predictors])
  roc_auc = roc_auc_score(y_val, y_pred_proba)
  accuracy = accuracy_score(y_val, y_pred)
  if accuracy > maxacc:
    accbest = i
    maxacc = accuracy
    print(f"Accuracy: {accuracy}")
  if roc_auc > maxroc:
    maxroc = roc_auc
    print(f"ROC AUC: {maxroc}")
    rocbest = i

In [ ]:
import optuna
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit

def objective(trial, X_train, y_train):
    # Suggest specific ranges for the trial
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 4), # Narrowed range
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.04), # Narrowed range
        'gamma': trial.suggest_float('gamma', 0.7, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.55, 0.7),
        'subsample': trial.suggest_float('subsample', 0.6, 0.75),
        'reg_lambda': trial.suggest_float('reg_lambda', 5, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1.5, 3),
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 4),
        'n_estimators': trial.suggest_int('n_estimators', 1800, 2200),
        # Static parameters
        'objective': 'binary:logistic',
        'random_state': 42,
        'eval_metric': "auc",
        'early_stopping_rounds': 50,
        'device': 'cuda',
        'n_jobs': -1,
    }

    # Use TimeSeriesSplit for cross-validation
    tscv = TimeSeriesSplit(n_splits=3)
    loglosses = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        # LogLoss is the negative log-loss; we want to minimize it.
        # Optuna minimizes the return value.
        loglosses.append(model.best_score)

    return np.mean(loglosses) # Return the average log-loss across all splits

# 3. Execution
study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: objective(trial, X[predictors], y), n_trials=1000)

print("Best hyperparameters found by Bayesian Optimization:")
print(study.best_params)

In [ ]:
best = study.best_params
best.update(
    {
        'objective': 'binary:logistic',
        'random_state': 42,
        'eval_metric': "logloss",
        'device': 'cuda',
        'n_jobs': -1,
    }
)
model2 = xgb.XGBClassifier(**best)
model2.fit(X[predictors], y)
# y_pred_proba = model2.predict_proba(future_games[predictors])[:, 1]
# y_pred_proba
y_pred = model2.predict_proba(future_X[future_X['home']==1][predictors])[:, 1]
future_X[future_X['home']==1]['target'] = y_pred
future_X[future_X['home']==1]

In [ ]:
import pickle
pd.set_option('display.max_columns', None)
with open("model.pkl", "wb") as f:
    pickle.dump(model2, f)

In [ ]:
with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

# Predict
y_pred = loaded_model.predict_proba(future_X[future_X['home']==1][predictors])[:, 1]

In [ ]:
from nba_api.stats.endpoints import leaguedashlineups
df = leaguedashlineups.LeagueDashLineups(season = '2024-25').get_data_frames()[0]
df[df['TEAM_ID']==1610612750].iloc[0]['GROUP_NAME']